# Databricks Unity Catalog - Table ACL

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/cross_demo_assets/Lakehouse_Demo_Team_architecture_2.png?raw=true" style="float: right" width="500px">

The main feature of Unity Catalog is to provide you an easy way to setup Table ACL (Access Control Level), but also build Dynamic Views based on each individual permission.

Typically, Analysts will only have access to customers from their country and won't be able to read GDPR/Sensitive informations (like email, firstname etc.)

A typical workflow in the Lakehouse architecture is the following:

* Data Engineers / Jobs can read and update the main data/schemas (ETL part)
* Data Scientists can read the final tables and update their features tables
* Data Analyst have READ access to the Data Engineering and Feature Tables and can ingest/transform additional data in a separate schema.
* Data is masked/anonymized dynamically based on each user access level

With Unity Catalog, your tables, users and groups are defined at the account level, cross workspaces. Ideal to deploy and operate a Lakehouse Platform across all your teams.

Let's see how this can be done with the Unity Catalog

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=governance&org_id=7474650537677415&notebook=%2F00-UC-Table-ACL&demo_name=uc-01-acl&event=VIEW&path=%2F_dbdemos%2Fgovernance%2Fuc-01-acl%2F00-UC-Table-ACL&version=1">

In [0]:
%run ./_resources/00-setup

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_uc_01_acl`


## Creating the CATALOG

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/uc/uc-base-1.png?raw=true" style="float: right" width="800px"/> 

The first step is to create a new catalog.

Unity Catalog works with 3 layers:

* CATALOG
* SCHEMA (or DATABASE)
* TABLE

To access one table, you can specify the full path: `SELECT * FROM &lt;CATALOG&gt;.&lt;SCHEMA&gt;.&lt;TABLE&gt;`

Note that the tables created before Unity Catalog are saved under the catalog named `hive_metastore`. Unity Catalog features are not available for this catalog.

Note that Unity Catalog comes in addition to your existing data, not hard change required!

In [0]:
%python
#The demo will create and use the catalog defined:
# see the catalog value in the ./config file
spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}');
#Make it default for future usage (we won't have to specify it)
spark.sql(f'USE CATALOG main');

In [0]:
-- the catalog has been created for your user and is defined as default. All shares will be created inside.
-- make sure you run the 00-setup cell above to init the catalog to your user. 
SELECT CURRENT_CATALOG();

current_catalog()
main


## Creating the SCHEMA
Next, we need to create the SCHEMA (or DATABASE).

Unity catalog provide the standard GRANT SQL syntax. We'll use it to GRANT CREATE and USAGE on our SCHEMA to all the users for this demo.

They'll be able to create extra table into this schema.

In [0]:
%python
# see schema value in the ./config file
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema}');
spark.sql(f'USE SCHEMA {schema}');

## Creating our table

We're all set! We can use standard SQL to create our tables.

We'll use a customers dataset, loading data about users (id, email etc...)

Because we want our demo to be available for all, we'll grant full privilege to the table to all USERS.

Note that the table owner is the current user. Owners have full permissions.<br/>
If you want to change the owner you can set it as following: ```ALTER TABLE <catalog>.uc_acl.customers OWNER TO `account users`;```

In [0]:
CREATE TABLE IF NOT EXISTS customers (
  id STRING,
  creation_date STRING,
  firstname STRING,
  lastname STRING,
  country STRING,
  email STRING,
  address STRING,
  gender DOUBLE,
  age_group DOUBLE);
-- GRANT SELECT, MODIFY on TABLE customers TO `account users`;  -- for the demo only, allow all users to edit the table - don't do that in production!

## Our customer data was filled for us!

The initialization cell already filled the table for us with fake data for the demo, let's review it's content.

In [0]:
SELECT * FROM  customers

name,ssn,region,id,firstname,lastname,email,address,canal,country,creation_date,last_activity_date,gender,age_group,churn
null,null,null,f120372e-022b-41b6-88f7-d6aec8565a6e,Christopher,Davis,sbrooks@hebert.com,"0866 Luna Crest Briggsside, AR 02197",WEBAPP,SPAIN,04-19-2013 00:00:00,06-04-2023 16:55:26,1.0,3.0,true
null,null,null,68eac5be-ba02-4b28-898b-f531b0e294f4,Andrea,Pierce,ybriggs@pierce.org,"4600 Richardson Dale New Wanda, NE 42581",MOBILE,USA,01-21-2014 00:00:00,06-06-2023 20:54:21,1.0,2.0,true
null,null,null,e4ac5e22-4112-4ade-be03-ebc9a4fcd971,Jane,Padilla,sellersmichael@mitchell.com,"286 Angela Row South Rachel, AL 90762",PHONE,USA,08-05-2021 00:00:00,06-02-2023 20:30:20,1.0,6.0,true
null,null,null,85b5cb37-703e-4b0b-9fd2-e26bfc28ebd6,Deanna,Brewer,wnunez@cole.com,"15141 Mendoza Brook Port Richardhaven, NE 17171",MOBILE,SPAIN,07-22-2021 00:00:00,06-01-2023 17:08:48,0.0,6.0,false
null,null,null,ae08a336-3126-4ef8-b8c0-2ca6b0277593,Carla,Harper,traceyhoward@zhang-gutierrez.org,"3583 Green Squares Port Jameshaven, MO 58447",MOBILE,FR,08-02-2021 00:00:00,06-04-2023 21:36:07,1.0,2.0,true
null,null,null,449a2bba-a29d-48dc-b623-2acd83dea4fb,Joel,Young,cathy85@silva.com,"16999 Michael Alley Apt. 910 Nguyenport, KS 99031",WEBAPP,FR,07-24-2021 00:00:00,06-02-2023 21:02:46,1.0,6.0,true
null,null,null,db8777a9-c7d8-4cae-9bac-80f4afd269b8,Ian,White,marie14@patrick-deleon.com,"123 Christian Village Suite 413 Port Ashleyport, VA 44207",WEBAPP,USA,08-14-2021 00:00:00,06-04-2023 16:39:48,1.0,4.0,false
null,null,null,7339fbd2-5b2b-4b9a-82c6-dc577ef0cf91,Elizabeth,Ortega,marshstacy@serrano.info,"58280 Brittany Terrace Apt. 046 Lake Pamelatown, PA 72400",PHONE,FR,07-24-2021 00:00:00,06-08-2023 08:04:54,1.0,6.0,false
null,null,null,65ae08c1-4327-47ce-b1e7-f88b196bf015,Dana,Walsh,veronica63@cardenas.com,Unit 4287 Box 8773 DPO AE 21291,MOBILE,FR,08-06-2021 00:00:00,06-04-2023 13:00:52,1.0,2.0,true
null,null,null,23671c7d-06b2-4a0d-a750-8faefd8348fe,James,Harris,aaronbrooks@combs.info,"57462 Duncan Land Stephenston, ND 98930",WEBAPP,SPAIN,07-23-2021 00:00:00,06-07-2023 03:58:15,1.0,8.0,true


## Granting users or group access

Let's now use Unity Catalog to GRANT permission on the table.

Unity catalog let you GRANT standard SQL permission to your objects, using the Unity Catalog users or group:

### Creating groups

Databricks groups can be created at the account level using the Account Admin UI, or the SCIM API. Here, we created the `dataengineers` group for this demo.

*Note on workspace-level groups: you can also create groups at a workspace level, however, we recommend managing permissions with UC at an account level.*

In [0]:
-- Let's grant all users a SELECT
-- GRANT SELECT ON TABLE customers TO `account users`; -- skip it for the demo, uncomment to make it available to all users!

-- We'll grant an extra MODIFY to our Data Engineer
-- Note: make sure you created the dataengineers group first as an account admin!
GRANT SELECT, MODIFY ON TABLE customers TO `dataengineers`;

In [0]:
SHOW GRANTS ON TABLE customers

Principal,ActionType,ObjectType,ObjectKey
cody.davis@databricks.com,ALL PRIVILEGES,CATALOG,main
aj.didonato@databricks.com,ALL PRIVILEGES,CATALOG,main
aj.didonato@databricks.com,MANAGE,CATALOG,main
pawanpreet.sangari@databricks.com,ALL PRIVILEGES,CATALOG,main
pawanpreet.sangari@databricks.com,MANAGE,CATALOG,main
076a8b71-b480-4fc6-82fb-c5dba4ae4343,SELECT,CATALOG,main
ANALYST_USA_DBDEMO,SELECT,TABLE,main.dbdemos_uc_01_acl.customers
aradhya.chouhan@databricks.com,ALL PRIVILEGES,CATALOG,main
kyra.wulffert@databricks.com,ALL PRIVILEGES,CATALOG,main
nan.chulpaibul@databricks.com,SELECT,CATALOG,main


## Conclusion

Unity Catalog gives you Table ACL permissions, leveraging users, group and table across multiple workspaces.

But UC not only gives you control over Tables. You can do more advanced permission and data access pattern such as dynamic masking at the row level.

### Next: Fine Grain Access control

Databricks Unity Catalog provides built-in capabilities to add dynamic masking on columns or rows.

Let's see how this can be done in the [01-Row-Column-access-control notebook ]($./01-Row-Column-access-control).